In [ ]:
# 1. Imports and config

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()   # adjust if 03_CRNN.ipynb is not in notebooks/
sys.path.append(str(PROJECT_ROOT))
print(PROJECT_ROOT)


D:\Curiously\Gsoc 2026\RenAIssance Project Test 1- Task 1 (Jupyter )\RenAIssance -  Test 2


In [2]:
from src.train import run_training
from src.data import build_vocab
from src.eval import evaluate_crnn


In [3]:
import yaml
from pathlib import Path
import torch

from src.train import run_training
from src.data import build_vocab
from src.eval import evaluate_crnn

config = yaml.safe_load(open("../configs/default.yaml"))
DATA_DIR = Path("..")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

In [4]:
from pathlib import Path
import os

print("CWD:", Path.cwd())
print("CSV exists here?:", Path("data/pages_gt_auto.csv").exists())


CWD: D:\Curiously\Gsoc 2026\RenAIssance Project Test 1- Task 1 (Jupyter )\RenAIssance -  Test 2\notebooks
CSV exists here?: False


In [5]:
os.chdir("..")        # or "..\\.." if 03_CRNN.ipynb is inside src/notebooks
print("New CWD:", Path.cwd())
print("CSV exists now?:", Path("data/pages_gt_auto.csv").exists())


New CWD: D:\Curiously\Gsoc 2026\RenAIssance Project Test 1- Task 1 (Jupyter )\RenAIssance -  Test 2
CSV exists now?: True


In [7]:
from src.train import run_training
model, char2idx, idx2char = run_training(config)



Epoch 1/30


Train loss: 1.8857 | Val loss: 0.0897
Saved best model to checkpoints\crnn_best.pt

Epoch 2/30


Train loss: 1.6742 | Val loss: 0.0892
Saved best model to checkpoints\crnn_best.pt

Epoch 3/30


Train loss: 1.5078 | Val loss: 0.0889
Saved best model to checkpoints\crnn_best.pt

Epoch 4/30


Train loss: 1.7862 | Val loss: 0.0879
Saved best model to checkpoints\crnn_best.pt

Epoch 5/30


...

Train loss: 1.3178 | Val loss: 0.0824
Saved best model to checkpoints\crnn_best.pt

Epoch 27/30


Train loss: 1.6269 | Val loss: 0.0829

Epoch 28/30


Train loss: 1.5474 | Val loss: 0.0822
Saved best model to checkpoints\crnn_best.pt

Epoch 29/30


Train loss: 1.6032 | Val loss: 0.0821
Saved best model to checkpoints\crnn_best.pt

Epoch 30/30


Train loss: 1.7208 | Val loss: 0.0820
Saved best model to checkpoints\crnn_best.pt


In [6]:
from torch.utils.data import DataLoader
from src.data import LineOCRDataset, collate_fn
from src.models import CRNN

# Load best checkpoint
ckpt = torch.load("checkpoints/crnn_best.pt", map_location=device)
idx2char = ckpt["idx2char"]
char2idx = ckpt["char2idx"]
config_loaded = ckpt["config"]

num_classes = len(char2idx)
model = CRNN(num_classes=num_classes).to(device)
model.load_state_dict(ckpt["model_state"])

# Build DataLoader for test set
from src.data import LineOCRDataset, collate_fn

test_ds = LineOCRDataset(config["data"]["test_csv"], char2idx,
                         img_height=config["data"]["img_height"])
test_loader = DataLoader(
    test_ds,
    batch_size=config["training"]["batch_size"],
    shuffle=False,
    num_workers=config["training"]["num_workers"],
    collate_fn=collate_fn,
)

metrics = evaluate_crnn(model, test_loader, idx2char, device)
print("CRNN test CER:", metrics["cer"])
print("CRNN test WER:", metrics["wer"])


CRNN test CER: 0.8315599365690116
CRNN test WER: 1.0211301404180655


In [7]:
from itertools import islice

model.eval()
examples = []

for batch in test_loader:
    images = batch["images"].to(device)
    texts = batch["texts"]
    paths = batch["paths"]

    logits = model(images)
    preds = model.decode_greedy(logits.cpu(), idx2char)

    for img_path, gt, pr in zip(paths, texts, preds):
        examples.append((img_path, gt, pr))
    break  # first batch only

for img_path, gt, pr in islice(examples, 10):
    print("Image:", img_path)
    print("GT   :", gt)
    print("CRNN :", pr)
    print("-" * 40)


Image: data\printed_pages\Buendia - Instruccion\Buendia - Instruccion_page_000.png
GT   : NOTES:		u and v are used interchangeably 	check against dictionary?
		f and s are used interchangeably 	check against d
CRNN :  e
----------------------------------------
Image: data\printed_pages\Buendia - Instruccion\Buendia - Instruccion_page_001.png
GT   : ictionary?
		accents are inconsistent 		should be ignored (except ñ)
		some letters have horizontal “cap”	tends to mean
CRNN : a e
----------------------------------------
Image: data\printed_pages\Buendia - Instruccion\Buendia - Instruccion_page_002.png
GT   : n follows, or ue after capped q
		some line end hyphens not present	leave words split for now, can decide later
		ç ol
CRNN : qqe
----------------------------------------
Image: data\printed_pages\Buendia - Instruccion\Buendia - Instruccion_page_003.png
GT   : d spelling is always modern z	teach AI to always interpret ç as z
PDF p2
Al
INFINITAMENTE AMABLE
NIÑO JESUS.
A Vos, Dul
CRNN :